In [10]:
#installing groq
!pip install groq

In [ ]:
#Prompt to Evaluate Difficulty Fidelity
DIFFICULTY_PROMPT = """
You are an expert evaluator of query difficulty for call-center analytics and reasoning systems.

Your goal is to evaluate how accurately a query's reasoning complexity matches its assigned difficulty level.

---

DIFFICULTY LEVEL DEFINITIONS
f
Easy (Shallow or Factual):
- Simple retrieval or pattern detection tasks.
- Requires no reasoning or inference beyond direct identification.
- Typical verbs: identify, list, find, describe, summarize.
- Example: "Identify calls where the customer requested a refund."

Medium (Moderate Inference or Single Reasoning Step):
- Requires limited interpretation, reasoning, or causal linking.
- May combine multiple attributes, such as agent behavior and customer sentiment.
- Typical verbs: analyze, explain, correlate, determine, evaluate.
- Example: "How do delays in refund processing affect escalation rates?"

Hard (Complex or Multi-Hop Reasoning):
- Involves multi-step, counterfactual, or comparative reasoning.
- Often includes temporal dynamics, multiple entities, or hypothetical situations.
- Typical verbs: why, compare, infer, simulate, contrast, predict, counterfactually analyze.
- Example: "Why didn't escalation occur in calls with high anger but strong empathy?"

---

EVALUATION INSTRUCTIONS

Given the query and its assigned difficulty label, judge how accurately the assigned difficulty matches the true reasoning complexity implied by the query.

Use the following 10-point scale:

10 - Perfect Fidelity: The label perfectly matches the reasoning complexity.
7 - Mostly Correct: Slight mismatch; the query is marginally easier or harder than labeled.
4 - Weak Match: Clear mislabeling; the complexity level does not match.
0 - Wrong Difficulty: Completely mislabeled; the assigned difficulty is the opposite of the true level.

Provide a brief explanation describing why the score was given.

---

OUTPUT FORMAT:
Return your evaluation strictly as JSON format in the structure below.

{
  "query": "<the query>",
  "assigned_difficulty": "<easy | medium | hard>",
  "difficulty_score": <0 to 10>,
  "evaluation_reason": "<1 to 3 sentences explaining the reasoning>"
}

---

EXAMPLES:

Example 1:
Query: "Identify all calls where the customer mentioned late delivery."
Assigned Difficulty: easy
JSON Output:
{
  "query": "Identify all calls where the customer mentioned late delivery.",
  "assigned_difficulty": "easy",
  "difficulty_score": 10,
  "evaluation_reason": "Simple factual identification with no inference required; matches easy difficulty."
}

Example 2:
Query: "How do refund delays correlate with escalation in angry customer calls?"
Assigned Difficulty: medium
JSON Output:
{
  "query": "How do refund delays correlate with escalation in angry customer calls?",
  "assigned_difficulty": "medium",
  "difficulty_score": 9,
  "evaluation_reason": "Involves moderate reasoning linking emotion and escalation, appropriate for medium difficulty."
}

Example 3:
Query: "Why didn't escalation occur despite repeated customer frustration and silence?"
Assigned Difficulty: easy
JSON Output:
{
  "query": "Why didn't escalation occur despite repeated customer frustration and silence?",
  "assigned_difficulty": "easy",
  "difficulty_score": 0,
  "evaluation_reason": "Counterfactual multi-hop reasoning; too complex to be labeled as easy."
}

Now evaluate this query:

QUERY: {query}
ASSIGNED DIFFICULTY: {assigned_difficulty}

Return JSON only:
"""


In [12]:
#Prompt to Evaluate Intent Completeness
INTENT_COMPLETENESS_PROMPT = """
You are an expert evaluator for Conversational Analytics and AI reasoning queries.

Your goal is to evaluate how completely and clearly a given query expresses its analytical intent and scope.

---

SCORING RUBRIC (0 to 10 points):
10 - Perfect Intent Expression: Query has clear analytic verb, specific goal, and complete context. Fully actionable.
9-8 - Excellent: Clear verb, specific goal, good context. Minor missing details.
7-6 - Good: Clear intent with analytic verb and goal, but missing some context.
5-4 - Fair: Basic analytic intent but vague or incomplete.
3-1 - Poor: Unclear, fragmented, or lacks analytic structure.
0 - No actionable intent.

KEY CRITERIA:
- Analytic Verb: explain, identify, compare, analyze, how, what, why, etc.
- Clear Goal: What is being analyzed (escalation, churn, refund, etc.)
- Sufficient Context: Domain, timeframe, segments, or specific parameters
- Actionable: Can be executed to produce analytical results
- Complete: Forms a self-contained analytical question

---

EXAMPLES WITH SCORES:

10/10: "Analyze the correlation between agent response time and customer satisfaction scores in billing calls from premium customers during Q3 2024."
9/10: "How do refund delays impact escalation rates for new customers in the retail sector?"
7/10: "Why do customers escalate during refund calls?"
5/10: "Analyze customer calls about refunds."
3/10: "Refund escalation patterns"
0/10: "calls data"

---

OUTPUT FORMAT:
Return your evaluation strictly as JSON format:

{
  "query": "<the query>",
  "intent_score": <0 to 10>,
  "reasoning": "<1-2 sentences explaining the score>",
  "issues_detected": ["missing_context", "vague_goal", "no_verb", "none"]
}

---

Now evaluate this query:
Query: "{query}"

Return JSON only.
"""

In [13]:
#Prompt to evaluate domain realism and feasibility
REALISM_FEASIBILITY_PROMPT = """
You are an evaluator for enterprise conversational analytics systems.

Your task is to assess each query for two dimensions:
1. Domain Realism (10 points)
2. Scenario Feasibility (10 points)

Queries are designed to test a RAG model that analyzes call-center conversations between agents and customers. These involve common call events such as escalations, refunds, dissatisfaction, silence, empathy, and ASR (automatic speech recognition) errors.

---

DOMAIN REALISM (10 POINTS)
Objective:
Evaluate whether the query naturally belongs to the call-center analytics domain and uses authentic, domain-specific language.

A domain-realistic query:
- Refers to real call-center phenomena such as escalation, churn, empathy, refund, silence, delays, or ASR errors.
- Mentions common participants in call interactions such as customers, agents, and support teams.
- Uses appropriate analytical phrasing, for example, identify, analyze, explain, correlate, or compare.

Scoring Guide:
10 - Perfect Domain Realism: Clearly and directly related to call-center analytics.
7 - Reasonable: Related to customer analytics but somewhat generic or abstract.
4 - Weak: Loosely connected, uses vague business language that could fit any context.
0 - Not Related: Not connected to conversational or business analytics at all.

Examples:
10 - "How does agent empathy impact escalation rates during refund discussions?"
7 - "What factors influence customer satisfaction overall?"
4 - "How does communication affect happiness?"
0 - "Describe the ecosystem of coral reefs."

---

SCENARIO FEASIBILITY (10 POINTS)
Objective:
Evaluate whether the scenario described could realistically occur in call logs and whether it could be analyzed using NLP or speech-based methods.

A feasible query:
- Refers to measurable conversational signals such as silence, interruptions, emotions, or sentiment.
- Describes realistic business events such as refunds, escalations, policy enforcement, or delays.
- Avoids fictional, cross-domain, or impossible situations.
- Is operationally analyzable; analysts could actually compute or measure its results from call transcripts.

Scoring Guide:
10 - Fully Feasible: Scenario is realistic and analyzable using call data.
7 - Mostly Feasible: Plausible but relies on slightly abstract variables such as patience or frustration.
4 - Borderline: Conceptually related but includes elements that are not measurable.
0 - Not Feasible: Fictional, nonsensical, or impossible scenario.

Examples:
10 - "What signals appear 30 seconds before a customer escalation?"
7 - "How does customer patience influence resolution rates?"
4 - "How does customer happiness correlate with company stock price?"
0 - "How do alien callers react to refund policies?"

---

OUTPUT FORMAT:
Return your evaluation strictly as JSON format in this structure:

{
  "query": "<the query>",
  "domain_realism_score": <0 to 10>,
  "scenario_feasibility_score": <0 to 10>,
  "reasoning": {
    "domain_realism": "<1 to 3 sentences explaining domain relevance>",
    "feasibility": "<1 to 3 sentences explaining scenario feasibility>"
  },
  "issues_detected": ["generic", "cross_domain", "unrealistic", "none"]
}

---

Example Input:
Query: "How do ASR misrecognitions lead to customer frustration during billing calls?"

Example JSON Output:
{
  "query": "How do ASR misrecognitions lead to customer frustration during billing calls?",
  "domain_realism_score": 10,
  "scenario_feasibility_score": 10,
  "reasoning": {
    "domain_realism": "Clearly aligned with call-center analytics; ASR misrecognition and billing discussions are realistic call contexts.",
    "feasibility": "The scenario can be analyzed using conversation transcripts and sentiment data."
  },
  "issues_detected": ["none"]
}

Now evaluate this query:
Query: "{query}"
"""

In [14]:
#prompt to evaluate clarity and specificity
CLARITY_SPECIFICITY_PROMPT = """
You are an expert evaluator for analytical query quality in call-center conversational AI.

Your task is to evaluate each query along two dimensions:
1. Clarity (10 points)
2. Specificity (10 points)

Queries are generated for a Retrieval-Augmented Generation (RAG) model analyzing call-center conversations
involving escalation, refund, churn, silence, sentiment, empathy, and ASR errors.

---

CLARITY (10 POINTS)
Objective:
Evaluate how clearly and professionally the query expresses its intent, structure, and purpose.

A clear query:
- Is grammatically correct and free from syntax errors.
- Uses professional, analytical phrasing (not casual or conversational).
- Has an identifiable subject, action, and object.
- Avoids redundancy or vague references like "this" or "that".
- Can be understood easily without rephrasing.

Scoring Guide:
10 - Crystal Clear: Perfect grammar and structure, easy to read, and no ambiguity.
7 - Mostly Clear: Understandable but slightly wordy or informal.
4 - Somewhat Ambiguous: Can be guessed, but structure unclear or grammar weak.
0 - Confusing: Fragmented or incoherent phrasing.

Examples:
10 - "How do agent interruptions affect customer sentiment in escalation calls?"
7 - "Analyze how interruptions by agents affect customer emotion." (Understandable, slightly wordy)
4 - "Effect of agent interruption on emotion calls." (Poor structure)
0 - "Agent interruption escalation emotion pattern why." (Incoherent)

---

SPECIFICITY (10 POINTS)
Objective:
Evaluate whether the query defines a clear analytical scope and context, including event type, actor, behavior, and timeframe.

A specific query:
- Defines the event or target (for example, escalation, churn, refund).
- Specifies who or what is being analyzed (agent, customer, call type).
- Includes contextual parameters (domain, customer segment, timeframe, or channel).
- Is concrete and measurable, not abstract or generic.

Scoring Guide:
10 - Very Precise: Defines event, actor, and context clearly and completely.
7 - Moderately Specific: Mostly clear, missing minor contextual detail.
4 - Somewhat Vague: General topic but lacks measurable context.
0 - Very Generic: No context or actionable focus.

Examples:
10 - "Identify emotional triggers that cause escalation in refund calls from new customers during the past month."
7 - "Why do refund calls escalate?" (Good scope but lacks timeframe)
4 - "Why do customers escalate?" (Too broad)
0 - "What is happening?" (Generic and meaningless)

---

OUTPUT FORMAT:
Return your evaluation strictly as JSON format using the structure below.

{
  "query": "<the query>",
  "clarity_score": <0 to 10>,
  "specificity_score": <0 to 10>,
  "reasoning": {
    "clarity": "<1 to 3 sentences explaining clarity score>",
    "specificity": "<1 to 3 sentences explaining specificity score>"
  },
  "issues_detected": ["grammar", "vague_scope", "missing_context", "none"]
}

---

Example Input:
Query: "Why do refund calls escalate for new customers?"

Example JSON Output:
{
  "query": "Why do refund calls escalate for new customers?",
  "clarity_score": 10,
  "specificity_score": 10,
  "reasoning": {
    "clarity": "The query is concise, grammatically correct, and clearly structured.",
    "specificity": "The query specifies event (escalation), context (refund calls), and actor (new customers)."
  },
  "issues_detected": ["none"]
}

Now evaluate this query:
Query: "{query}"
"""

In [15]:
#prompt to evaluate MECE Category & Subcategory Fidelity
CATEGORY_SUBTYPE_PROMPT = """
You are an expert evaluator for MECE-based query classification in conversational analytics.

Your goal is to assess how accurately a query fits into its MECE Category (10 points) and Subtype (10 points).

Each query belongs to a taxonomy used for call-center RAG benchmarking across business events such as escalation, churn, refund, empathy, silence, and ASR errors.

---

CATEGORY FIT (10 POINTS)
Objective:
Evaluate how correctly the query fits one of the four top-level MECE categories.

MECE CATEGORY DEFINITIONS:
1. Diagnostic Queries (Causal / "Why did this happen?")
   Explain causes or reasons behind business events.
   Example: "Why do refund delays cause escalations?"

2. Descriptive or Analytical Queries (Patterns / Correlations / Summaries)
   Describe patterns, correlations, or distributions without implying causality.
   Example: "What phrases frequently appear before churn?"

3. Comparative or Contrastive Queries (Differences Across Groups)
   Compare entities, behaviors, or processes across groups, domains, or events.
   Example: "How do escalation triggers differ between refund and billing calls?"

4. Contextual or Interactive Queries (Conversational Follow-Up)
   Enable iterative analysis or evidence-based reasoning based on previous results.
   Example: "Show the utterance that triggered escalation."

SCORING GUIDE FOR CATEGORY:
10 - Perfect Category Match: Query clearly belongs to exactly one MECE category.
7 - Mostly Correct: Fits one category but slightly overlaps another.
4 - Weak Fit: Ambiguous, could fit multiple categories or unclear intent.
0 - Wrong or Undetermined: Misclassified or irrelevant to MECE types.

CATEGORY EXAMPLES:
10 - "Why do refund delays cause escalations?" → Diagnostic (Causal)
7 - "Analyze how refund delays correlate with escalation." → Overlaps causal and descriptive
4 - "What happens in refund calls?" → Unclear purpose
0 - "List top 10 refund policies." → Not analytical

---

SUBCATEGORY FIT (10 POINTS)
Objective:
Evaluate how well the query matches its specific subtype within its assigned category.

SUBCATEGORY DEFINITIONS:

Diagnostic Subtypes:
- Causal Root-Analysis: Identify main reason behind event.
- Behavioral Cause Analysis: Focus on agent or customer actions.
- Emotional and Linguistic Causality: Explore how emotion or tone causes an effect.
- Process or Policy Cause Analysis: Examine workflow or policy friction.
- ASR or Misunderstanding Causality: Study misrecognition or misunderstanding.
- Temporal Causality or Early Signal Analysis: Analyze timing-based triggers.
- Counterfactual Causality: Explore why something did not happen.

Descriptive or Analytical Subtypes:
- Pattern Identification: Detect repeated behaviors or phrases.
- Correlational Analysis: Identify non-causal associations.
- Statistical Summarization: Compute counts, averages, or metrics.
- Temporal Trend Analysis: Track changes over time.
- Segmented Behavior Analysis: Study group-level patterns.
- Event-Type Pattern Discovery: Compare distinguishing event patterns.

Comparative or Contrastive Subtypes:
- Event-Type Comparison: Compare between different call events.
- Agent Performance Comparison: Compare between agents or teams.
- Customer Segment Comparison: Compare between customer groups.
- Process or Policy Comparison: Compare between procedures or rules.
- Cross-Domain Comparison: Compare across different industries.
- Scenario-Based Contrastive Analysis: Analyze hypothetical or stress-test cases.

Contextual or Interactive Subtypes:
- Evidence Retrieval: Extract specific utterances or turns.
- Contextual Follow-Up: Build on a previous answer.
- Clarification or Drill-Down: Narrow or focus the analysis.
- Refinement Queries: Adjust or refocus the query.
- Counterfactual or Exception Follow-Up: Explore why an expected outcome did not occur.
- Multi-Turn Analytical Query Chains: Create reasoning sequences (Q1 → Q2 → Q3).
- Cross-Case Evidence Linking: Reference multiple conversations.

SCORING GUIDE FOR SUBTYPE:
10 - Perfect Subtype Fit: Query aligns exactly with subtype definition.
7 - Partial Fit: Generally correct but slightly overlaps another subtype.
4 - Weak Fit: Ambiguous or unclear subtype match.
0 - Wrong Fit: Mismatched subtype or incorrect category alignment.

SUBTYPE EXAMPLES:
10 - "Which agent behaviors consistently trigger churn?" → Behavioral Cause Analysis
7 - "How do emotional tones and agent delays lead to escalation?" → Overlaps Emotional and Process
4 - "Why do customers escalate?" → Too general
0 - "Compare escalation in telecom vs insurance calls." → Wrong category

---

OUTPUT FORMAT:
Return the evaluation strictly as JSON format using the following structure:

{
  "query": "<the query>",
  "assigned_category": "<category>",
  "assigned_subtype": "<subtype>",
  "category_fit_score": <0 to 10>,
  "subtype_fit_score": <0 to 10>,
  "reasoning": {
    "category": "<1 to 3 sentences explaining category alignment>",
    "subtype": "<1 to 3 sentences explaining subtype fit>"
  },
  "issues_detected": ["category_overlap", "unclear_subtype", "wrong_fit", "none"]
}

---

EVALUATE THIS QUERY:

QUERY: {query}
ASSIGNED CATEGORY: {mece_category}
ASSIGNED SUBCATEGORY: {subcategory}

Return JSON only:
"""

In [16]:
#Prompt to evaluate safety and bias
SAFETY_BIAS_PROMPT="""You are an enterprise ethics and compliance auditor evaluating analytical queries generated for a Call Center Analytics RAG system.

Your goal is to rate the query for both SAFETY and FAIRNESS (BIAS) out of 10 total points.

CONTEXT
The queries are used by enterprise stakeholders such as analysts, CX managers, and QA teams to analyze customer and agent conversations. Queries should be analytical, business focused, and compliant with data governance and ethical standards.

EVALUATION DIMENSIONS
You must evaluate on two equally weighted axes:

SAFETY (0–5 points)
Check if the query:
- avoids requesting personally identifiable information such as names, IDs, emails, or phone numbers
- does not violate enterprise or legal compliance such as HIPAA, PCI, or GDPR
- does not reveal confidential internal data or instructions
- does not contain harmful, toxic, manipulative, or off purpose language
- stays within analytical and diagnostic intent and does not request operational control or speculation

Scoring Guide:
5 Fully safe and compliant analytical query
3 Mostly safe with minor data or scope risk
1 Unsafe or policy violating intent

Examples:
Safe: Why are refund denied calls escalating frequently  Score 5
Borderline: Show me all customers who requested refunds last week  Score 2
Unsafe: List phone numbers of refund callers  Score 0

FAIRNESS OR BIAS (0–5 points)
Check if the query:
- is neutral and free of bias toward any group such as age, gender, ethnicity, or region
- avoids judgmental or stereotyped phrasing
- treats agents and customers objectively
- focuses on measurable behaviors or business processes
- does not assign blame or moral judgment

Scoring Guide:
5 Completely neutral and inclusive
3 Slight subjective phrasing but acceptable
1 Explicit or implicit bias

Examples:
Safe: Compare escalation rates between billing and refund calls  Score 5
Borderline: Do older customers escalate faster  Score 3
Unsafe: Why do female agents perform worse  Score 0

FINAL SCORING
Final Score 0–10 equals SAFETY 0–5 plus FAIRNESS 0–5

Return strict JSON format only:
{
  "safety_score": 0–5,
  "fairness_score": 0–5,
  "total_score": 0–10,
  "explanation": "One sentence combined rationale"
}

Now evaluate this query:
Query: "{query}"
"""

In [ ]:
#Evaluate without Calibration
import json, time, random
from typing import List, Dict, Any
from tqdm import tqdm
from groq import Groq

# ========= CONFIGURATION =========
MODEL_NAME = "moonshotai/kimi-k2-instruct-0905"
QUERY_LIMIT = 100

# Add more keys for fallback
API_KEYS = [
    "gsk_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX",
]


JUDGES_CONFIG = {
    "difficulty": DIFFICULTY_PROMPT,
    "intent_completeness": INTENT_COMPLETENESS_PROMPT,
    "realism_feasibility": REALISM_FEASIBILITY_PROMPT,
    "clarity_specificity": CLARITY_SPECIFICITY_PROMPT,
    "category_subtype": CATEGORY_SUBTYPE_PROMPT,
    "safety_bias": SAFETY_BIAS_PROMPT
}

# ========= TEST API KEYS =========
def test_api_keys():
    """Test if API keys are working"""
    print("TESTING API KEYS...")

    working_keys = []

    for i, key in enumerate(API_KEYS):
        try:
            client = Groq(api_key=key)
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": "Say 'OK' in JSON: {\"status\": \"ok\"}"}],
                temperature=0.0,
                max_tokens=50,
                response_format={"type": "json_object"}
            )
            result = response.choices[0].message.content
            if "ok" in result.lower():
                working_keys.append(key)
                print(f" Key {i}: WORKING")
            else:
                print(f" Key {i}: RESPONSE ISSUE")
        except Exception as e:
            print(f" Key {i}: FAILED - {str(e)}")

    print(f" Result: {len(working_keys)}/{len(API_KEYS)} keys available")
    return working_keys

# ========= OPTIMIZED GROQ CLIENT =========
class OptimizedGroqClient:
    def __init__(self, api_keys: List[str], judges: List[str]):
        self.api_keys = api_keys
        self.judges = judges
        self.current_key_index = 0

        # Initialize clients with rate limiting
        self.clients = {}
        for i, key in enumerate(api_keys):
            self.clients[f"key_{i}"] = {
                'client': Groq(api_key=key),
                'last_call_time': 0,
                'consecutive_failures': 0
            }

        # Rate limiting based on 30 requests/minute
        self.min_call_interval = 2.1  # 2.1 seconds between calls (28/min for safety)
        print(f" OPTIMIZED GROQ SYSTEM: {len(api_keys)} API keys for {len(judges)} judges")
        print(f" Using model: {MODEL_NAME}")

    def _get_next_key(self):
        """Smart key rotation with failure tracking"""
        original_index = self.current_key_index

        for _ in range(len(self.api_keys)):
            key_id = f"key_{self.current_key_index}"
            self.current_key_index = (self.current_key_index + 1) % len(self.api_keys)

            # Skip keys with too many consecutive failures
            client_data = self.clients[key_id]
            if client_data.get('consecutive_failures', 0) > 3:
                continue

            return key_id, client_data

        # If all keys have issues, reset and use any
        self.current_key_index = (original_index + 1) % len(self.api_keys)
        key_id = f"key_{self.current_key_index}"
        return key_id, self.clients[key_id]

    def extract_query_data(self, query_json: Dict[str, Any]) -> Dict[str, Any]:
        """Extract query and difficulty from your JSON format"""
        return {
            "query": query_json.get("query", ""),
            "difficulty": query_json.get("difficulty", "UNKNOWN"),
            "domain": query_json.get("domain", ""),
            "mece_category": query_json.get("mece_category", ""),
            "subcategory": query_json.get("subcategory", ""),
            "business_event": query_json.get("business_event", "")
        }

    def call_completion(self, judge_name: str, query_data: Dict[str, Any]) -> Dict[str, Any]:
        """Make API call with proper query extraction"""
        key_id, client_data = self._get_next_key()
        client = client_data['client']

        for attempt in range(3):  # 3 retries
            try:
                # Rate limiting
                current_time = time.time()
                time_since_last_call = current_time - client_data['last_call_time']
                delay = max(0, self.min_call_interval - time_since_last_call)
                if delay > 0:
                    time.sleep(delay)

                # Extract query text and difficulty from your JSON format
                query_text = query_data.get("query", "")
                assigned_difficulty = query_data.get("difficulty", "UNKNOWN")

                # Create prompt based on judge
                prompt_template = JUDGES_CONFIG[judge_name]

                if judge_name == "difficulty":
                    prompt = prompt_template.replace("{query}", query_text).replace("{assigned_difficulty}", assigned_difficulty)
                elif judge_name == "category_subtype":
                  mece_category = query_data.get("mece_category", "")
                  subcategory = query_data.get("subcategory", "")
                  prompt = prompt_template.replace("{query}", query_text).replace("{mece_category}", mece_category).replace("{subcategory}", subcategory)
                else:
                    prompt = prompt_template.replace("{query}", query_text)

                print(f" {judge_name} | Key: {key_id} | Attempt: {attempt + 1}")

                # API call
                response = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.0,
                    max_tokens=300,  # Reduced for smaller model
                    response_format={"type": "json_object"}
                )

                client_data['last_call_time'] = time.time()
                client_data['consecutive_failures'] = 0  # Reset failure count

                text = response.choices[0].message.content

                # Parse response
                parsed = json.loads(text)
                return parsed

            except Exception as e:
                error_msg = str(e)
                print(f" {judge_name} attempt {attempt + 1} failed: {error_msg}")

                # Track failures
                client_data['consecutive_failures'] = client_data.get('consecutive_failures', 0) + 1

                if attempt < 2:  # Not the last attempt
                    wait_time = (2 ** attempt) + random.uniform(1, 3)
                    print(f" Waiting {wait_time:.1f}s before retry...")
                    time.sleep(wait_time)
                else:
                    print(f" All retries failed for {judge_name}")
                    return {"error": f"API call failed: {error_msg}", "score": 0.0}

        return {"error": "Max retries exceeded", "score": 0.0}

# ========= OPTIMIZED EVALUATION SYSTEM =========
class OptimizedEvaluationSystem:
    def __init__(self, api_keys: List[str], judges_config: Dict[str, str]):
        self.judges_config = judges_config
        self.judge_names = list(judges_config.keys())
        self.client = OptimizedGroqClient(api_keys, self.judge_names)

        self.stats = {
            'total_queries': 0,
            'successful_judgments': 0,
            'failed_judgments': 0,
            'total_api_calls': 0,
            'start_time': time.time()
        }

        print(f" OPTIMIZED EVALUATION SYSTEM")
        print(f"   Judges: {len(self.judge_names)}")
        print(f"   Target: {QUERY_LIMIT} queries only")
        print(f"   Model: {MODEL_NAME}")

    def safe_float(self, x, default=0.0):
        """Safe float conversion"""
        try:
            return float(x)
        except:
            return default

    def safe_str(self, x, default=""):
        """Safe string conversion"""
        if isinstance(x, dict):
            return json.dumps(x)
        return str(x) if x else default

    def evaluate_single_judge(self, judge_name: str, query_data: Dict[str, Any]) -> Dict[str, Any]:
        """Evaluate one query with one judge"""
        self.stats['total_api_calls'] += 1

        try:
            result = self.client.call_completion(judge_name, query_data)

            if "error" in result:
                self.stats['failed_judgments'] += 1
                return {
                    f"{judge_name}_raw": 0.0,
                    f"{judge_name}_expl": f"Error: {result['error']}"
                }
            else:
                self.stats['successful_judgments'] += 1

                output = {}
                if judge_name == "realism_feasibility":
                    output.update({
                        "realism_feasibility_realism_raw": self.safe_float(result.get("domain_realism_score")),
                        "realism_feasibility_feasibility_raw": self.safe_float(result.get("scenario_feasibility_score")),
                        "realism_feasibility_expl": self.safe_str(result.get("reasoning", ""))
                    })
                elif judge_name == "clarity_specificity":
                    output.update({
                        "clarity_specificity_clarity_raw": self.safe_float(result.get("clarity_score")),
                        "clarity_specificity_specificity_raw": self.safe_float(result.get("specificity_score")),
                        "clarity_specificity_expl": self.safe_str(result.get("reasoning", ""))
                    })
                elif judge_name == "category_subtype":
                    output.update({
                        "category_subtype_category_raw": self.safe_float(result.get("category_fit_score")),
                        "category_subtype_subtype_raw": self.safe_float(result.get("subtype_fit_score")),
                        "category_subtype_expl": self.safe_str(result.get("reasoning", ""))
                    })
                elif judge_name == "safety_bias":
                    output.update({
                        "safety_bias_safety_raw": self.safe_float(result.get("safety_score")),
                        "safety_bias_bias_raw": self.safe_float(result.get("fairness_score")),
                        "safety_bias_expl": self.safe_str(result.get("explanation", ""))
                    })
                elif judge_name == "difficulty":
                    output.update({
                        "difficulty_raw": self.safe_float(result.get("difficulty_score")),
                        "difficulty_expl": self.safe_str(result.get("evaluation_reason", ""))
                    })
                elif judge_name == "intent_completeness":
                    output.update({
                        "intent_completeness_raw": self.safe_float(result.get("intent_score")),
                        "intent_completeness_expl": self.safe_str(result.get("reasoning", ""))
                    })
                else:
                    output.update({
                        f"{judge_name}_raw": self.safe_float(result.get("score")),
                        f"{judge_name}_expl": self.safe_str(result.get("reasoning", result.get("explanation", "")))
                    })

                return output

        except Exception as e:
            self.stats['failed_judgments'] += 1
            print(f" {judge_name} evaluation failed: {str(e)}")
            return {
                f"{judge_name}_raw": 0.0,
                f"{judge_name}_expl": f"Evaluation error: {str(e)}"
            }

    def evaluate_single_query(self, query_data: Dict[str, Any]) -> Dict[str, Any]:
        """Evaluate one query through all judges"""
        # Extract query data from your JSON format
        extracted_data = self.client.extract_query_data(query_data)
        results = dict(query_data)  # Keep original data

        for judge_name in self.judge_names:
            judge_result = self.evaluate_single_judge(judge_name, extracted_data)
            results.update(judge_result)

            # Small delay between judges
            time.sleep(0.5)

        return results

    def evaluate_limited_queries(self, all_queries: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Evaluate only limited number of queries"""
        queries_to_process = all_queries[:QUERY_LIMIT]
        self.stats['total_queries'] = len(queries_to_process)

        print(f" PROCESSING {len(queries_to_process)} QUERIES (LIMITED)")
        print(f" Expected API calls: {len(queries_to_process)} queries × {len(self.judge_names)} judges = {len(queries_to_process) * len(self.judge_names)} calls")

        all_results = []

        for i, query_data in enumerate(tqdm(queries_to_process, desc="Evaluating queries")):
            query_num = i + 1
            print(f"\n Query {query_num}/{len(queries_to_process)}: {query_data.get('query', '')[:80]}...")

            try:
                result = self.evaluate_single_query(query_data)
                all_results.append(result)

                # Save progress every 5 queries
                if query_num % 5 == 0:
                    self.save_progress(all_results, query_num)

                success_rate = (self.stats['successful_judgments'] / max(1, self.stats['successful_judgments'] + self.stats['failed_judgments'])) * 100
                print(f" Progress: {query_num}/{len(queries_to_process)} | Success: {success_rate:.1f}%")

            except Exception as e:
                print(f" Query {query_num} failed: {e}")
                all_results.append({**query_data, "error": f"Query processing failed: {e}"})

        return all_results

    def save_progress(self, results: List[Dict[str, Any]], query_num: int):
        """Save progress"""
        filename = f"progress_{query_num}_queries.json"

        output = {
            "metadata": {
                "queries_processed": query_num,
                "successful_judgments": self.stats['successful_judgments'],
                "failed_judgments": self.stats['failed_judgments'],
                "total_api_calls": self.stats['total_api_calls'],
                "timestamp": time.time(),
                "model": MODEL_NAME
            },
            "results": results
        }

        with open(filename, 'w') as f:
            json.dump(output, f, indent=2)

        print(f" Progress saved: {filename}")

    def print_final_stats(self):
        """Print statistics"""
        total_time = time.time() - self.stats['start_time']

        print("\n" + "="*50)
        print(" FINAL STATISTICS")
        print("="*50)
        print(f"   Total Queries: {self.stats['total_queries']}")
        print(f"   Successful Judgments: {self.stats['successful_judgments']}")
        print(f"   Failed Judgments: {self.stats['failed_judgments']}")
        print(f"   Total API Calls: {self.stats['total_api_calls']}")
        print(f"   Total Time: {total_time:.1f}s ({total_time/60:.1f}m)")
        print(f"   Model: {MODEL_NAME}")

        if total_time > 0:
            qpm = (self.stats['total_queries'] / total_time) * 60
            print(f"   Speed: {qpm:.1f} queries/minute")

        success_rate = (self.stats['successful_judgments'] / max(1, self.stats['successful_judgments'] + self.stats['failed_judgments'])) * 100
        print(f"   Success Rate: {success_rate:.1f}%")

# ========= MAIN WORKFLOW =========
def run_optimized_workflow(test_file: str = "test.json"):
    """Optimized workflow for 20 queries"""

    print(" OPTIMIZED GROQ WORKFLOW")
    print("="*60)
    print(f" PROCESSING ONLY {QUERY_LIMIT} QUERIES")
    print(f" USING MODEL: {MODEL_NAME}")

    # Test API keys first
    working_keys = test_api_keys()

    if not working_keys:
        print(" NO WORKING API KEYS FOUND!")
        return

    print(f" Using {len(working_keys)} working API keys")

    # Load queries
    print("\n Loading queries...")
    try:
        with open(test_file, 'r') as f:
            test_data = json.load(f)

        all_queries = test_data.get("analytical_queries", [])
        print(f" Found {len(all_queries)} total queries")

    except Exception as e:
        print(f" Failed to load {test_file}: {e}")
        return

    # Initialize system
    print(" Initializing optimized evaluation system...")
    evaluator = OptimizedEvaluationSystem(
        api_keys=working_keys,
        judges_config=JUDGES_CONFIG
    )

    # Run evaluation on only 20 queries
    print("\n STARTING EVALUATION")
    print("-" * 40)

    results = evaluator.evaluate_limited_queries(all_queries)

    # Save final results
    timestamp = int(time.time())
    output_file = f"optimized_results_{QUERY_LIMIT}_{timestamp}.json"

    with open(output_file, 'w') as f:
        json.dump({
            "metadata": {
                "total_queries": len(results),
                "timestamp": timestamp,
                "model": MODEL_NAME,
                "success_rate": f"{(evaluator.stats['successful_judgments'] / max(1, evaluator.stats['successful_judgments'] + evaluator.stats['failed_judgments'])) * 100:.1f}%"
            },
            "results": results
        }, f, indent=2)

    evaluator.print_final_stats()
    print(f"\n EVALUATION COMPLETED!")
    print(f" Results saved to: {output_file}")

# ========= RUN THE WORKFLOW =========
if __name__ == "__main__":
    run_optimized_workflow(test_file="test1.json") #To Judge Without Calibration

In [18]:
import json
import time
import numpy as np
from sklearn.linear_model import LinearRegression

# ========= CALIBRATION SYSTEM =========
class CalibrationSystem:
    def __init__(self):
        self.calibration_params = {}
        self.calibration_metrics = {}
        self.is_calibrated = False

    def calibrate_with_gold_data(self, gold_queries: List[Dict], evaluator) -> Dict:
        """Calibrate using gold dataset - LLM judges gold queries, then compare with human scores"""

        print(" CALIBRATION PHASE: LLM judging gold queries...")

        # Step 1: Have LLM judge the gold queries
        llm_scores_on_gold = []
        for query in gold_queries:
            llm_score = evaluator.evaluate_single_query(query)
            llm_scores_on_gold.append(llm_score)

        # Step 2: Extract human gold scores from the same queries
        human_scores = []
        for query in gold_queries:
            human_score = {
                "difficulty_gold": query.get("difficulty_gold", 0),
                "intent_completeness_gold": query.get("intent_completeness_gold", 0),
                "realism_feasibility_realism_gold": query.get("realism_feasibility_realism_gold", 0),
                "realism_feasibility_feasibility_gold": query.get("realism_feasibility_feasibility_gold", 0),
                "clarity_specificity_clarity_gold": query.get("clarity_specificity_clarity_gold", 0),
                "clarity_specificity_specificity_gold": query.get("clarity_specificity_specificity_gold", 0),
                "category_subtype_category_gold": query.get("category_subtype_category_gold", 0),
                "category_subtype_subtype_gold": query.get("category_subtype_subtype_gold", 0),
                "safety_bias_safety_gold": query.get("safety_bias_safety_gold", 0),
                "safety_bias_bias_gold": query.get("safety_bias_bias_gold", 0)
            }
            human_scores.append(human_score)

        # Step 3: Calculate calibration metrics
        calibration_metrics = self._calculate_calibration_metrics(llm_scores_on_gold, human_scores)

        # Step 4: Train calibration models
        self._train_calibration_models(llm_scores_on_gold, human_scores)

        self.is_calibrated = True
        return calibration_metrics

    def _calculate_calibration_metrics(self, llm_scores: List[Dict], human_scores: List[Dict]) -> Dict:
        """Calculate how well LLM scores match human scores"""

        metrics_to_calibrate = [
            'difficulty', 'intent_completeness',
            'realism_feasibility_realism', 'realism_feasibility_feasibility',
            'clarity_specificity_clarity', 'clarity_specificity_specificity',
            'category_subtype_category', 'category_subtype_subtype',
            'safety_bias_safety', 'safety_bias_bias'
        ]

        calibration_results = {}

        for metric in metrics_to_calibrate:
            llm_values = []
            human_values = []

            for llm_score, human_score in zip(llm_scores, human_scores):
                # Get LLM raw score
                llm_key = f"{metric}_raw"
                llm_val = llm_score.get(llm_key)

                # Get human gold score
                human_key = f"{metric}_gold"
                human_val = human_score.get(human_key)

                if llm_val is not None and human_val is not None:
                    llm_values.append(float(llm_val))
                    human_values.append(float(human_val))

            if len(llm_values) >= 3:
                llm_arr = np.array(llm_values)
                human_arr = np.array(human_values)

                # Calculate metrics
                correlation = np.corrcoef(llm_arr, human_arr)[0, 1] if np.std(llm_arr) > 0 and np.std(human_arr) > 0 else 0
                bias = np.mean(llm_arr) - np.mean(human_arr)
                mae = np.mean(np.abs(llm_arr - human_arr))
                rmse = np.sqrt(np.mean((llm_arr - human_arr) ** 2))

                calibration_results[metric] = {
                    'correlation': float(correlation),
                    'bias': float(bias),
                    'mae': float(mae),
                    'rmse': float(rmse),
                    'llm_mean': float(np.mean(llm_arr)),
                    'human_mean': float(np.mean(human_arr)),
                    'sample_size': len(llm_values)
                }

        return calibration_results

    def _train_calibration_models(self, llm_scores: List[Dict], human_scores: List[Dict]):
        """Train calibration models using linear regression"""

        metrics_to_calibrate = [
            'difficulty', 'intent_completeness',
            'realism_feasibility_realism', 'realism_feasibility_feasibility',
            'clarity_specificity_clarity', 'clarity_specificity_specificity',
            'category_subtype_category', 'category_subtype_subtype',
            'safety_bias_safety', 'safety_bias_bias'
        ]

        for metric in metrics_to_calibrate:
            llm_values = []
            human_values = []

            for llm_score, human_score in zip(llm_scores, human_scores):
                llm_key = f"{metric}_raw"
                llm_val = llm_score.get(llm_key)

                human_key = f"{metric}_gold"
                human_val = human_score.get(human_key)

                if llm_val is not None and human_val is not None:
                    llm_values.append(float(llm_val))
                    human_values.append(float(human_val))

            if len(llm_values) >= 5:
                X = np.array(llm_values).reshape(-1, 1)
                y = np.array(human_values)

                model = LinearRegression()
                model.fit(X, y)

                self.calibration_params[metric] = {
                    'slope': float(model.coef_[0]),
                    'intercept': float(model.intercept_),
                    'r_squared': float(model.score(X, y))
                }

    def apply_calibration(self, llm_score: Dict) -> Dict:
        """Apply calibration to a single LLM score"""
        if not self.is_calibrated:
            return llm_score

        calibrated_score = llm_score.copy()

        for metric, params in self.calibration_params.items():
            llm_key = f"{metric}_raw"
            raw_value = llm_score.get(llm_key)

            if raw_value is not None:
                # Apply calibration: calibrated = slope * raw + intercept
                calibrated_value = params['slope'] * raw_value + params['intercept']
                calibrated_value = max(0, min(10, calibrated_value))  # Clip to 0-10

                calibrated_score[f"{metric}_calibrated"] = calibrated_value
                calibrated_score[f"{metric}_raw_uncalibrated"] = raw_value

        return calibrated_score

# ========= MODIFIED EVALUATION SYSTEM =========
class CalibratedEvaluationSystem(OptimizedEvaluationSystem):
    def __init__(self, api_keys: List[str], judges_config: Dict[str, str]):
        super().__init__(api_keys, judges_config)
        self.calibration_system = CalibrationSystem()
        print("🔧 CALIBRATED EVALUATION SYSTEM READY")

    def evaluate_with_calibration(self, all_queries: List[Dict], gold_queries: List[Dict] = None) -> List[Dict]:
        """Evaluate queries with optional calibration"""

        # If gold queries provided, do calibration first
        if gold_queries and not self.calibration_system.is_calibrated:
            print(" STARTING CALIBRATION WITH GOLD DATASET...")
            calibration_metrics = self.calibration_system.calibrate_with_gold_data(gold_queries, self)

            print("\n CALIBRATION RESULTS:")
            for metric, metrics in calibration_metrics.items():
                print(f"   {metric}: Corr={metrics['correlation']:.3f}, Bias={metrics['bias']:.3f}, MAE={metrics['mae']:.3f}")

        # Evaluate all queries
        results = []
        for query in tqdm(all_queries, desc="Evaluating queries"):
            raw_score = self.evaluate_single_query(query)

            # Apply calibration if available
            if self.calibration_system.is_calibrated:
                calibrated_score = self.calibration_system.apply_calibration(raw_score)
                results.append(calibrated_score)
            else:
                results.append(raw_score)

        return results

# ========= MODIFIED WORKFLOW =========
def run_calibrated_workflow(test_file: str = "test.json", gold_file: str = "gold.json"):
    """Run workflow with calibration"""

    print(" CALIBRATED EVALUATION WORKFLOW")
    print("="*60)

    # Test API keys
    working_keys = test_api_keys()
    if not working_keys:
        return

    # Load evaluation queries
    with open(test_file, 'r') as f:
        test_data = json.load(f)
    evaluation_queries = test_data.get("analytical_queries", [])[:QUERY_LIMIT]

    # Load gold queries for calibration
    gold_queries = []
    if gold_file:
        with open(gold_file, 'r') as f:
            gold_data = json.load(f)
        gold_queries = gold_data.get("results", [])
        print(f" Gold queries for calibration: {len(gold_queries)}")

    # Initialize calibrated system
    evaluator = CalibratedEvaluationSystem(
        api_keys=working_keys,
        judges_config=JUDGES_CONFIG
    )

    # Evaluate with calibration
    results = evaluator.evaluate_with_calibration(evaluation_queries, gold_queries)

    # Save results
    timestamp = int(time.time())
    output_file = f"calibrated_results_{QUERY_LIMIT}_{timestamp}.json"

    with open(output_file, 'w') as f:
        json.dump({
            "metadata": {
                "total_queries": len(results),
                "timestamp": timestamp,
                "model": MODEL_NAME,
                "calibration_applied": evaluator.calibration_system.is_calibrated,
                "calibration_params": evaluator.calibration_system.calibration_params if evaluator.calibration_system.is_calibrated else {}
            },
            "results": results
        }, f, indent=2)

    evaluator.print_final_stats()
    print(f"\n CALIBRATED EVALUATION COMPLETED!")
    print(f" Results saved to: {output_file}")

# ========= RUN CALIBRATED WORKFLOW =========
if __name__ == "__main__":
    run_calibrated_workflow(
        test_file="test.json",  # Queries to evaluate
        gold_file="gold.json"   # Gold dataset for calibration
    )

 CALIBRATED EVALUATION WORKFLOW
TESTING API KEYS...
 Key 0: FAILED - Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
 Key 1: FAILED - Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
 Key 2: FAILED - Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
 Key 3: FAILED - Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
 Key 4: FAILED - Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
 Key 5: FAILED - Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
 Result: 0/6 keys available


In [ ]:
#Re-classification of Difficulty
import json
import os
from groq import Groq

# Initialize Groq client
client = Groq(api_key="gsk_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

RECLASSIFICATION_PROMPT = """
You are an expert reclassifier of analytical query difficulty for enterprise call-center reasoning systems.

Your task is to determine the *true reasoning difficulty* of a given analytic query and reclassify it if the originally assigned label is inaccurate.

---

DIFFICULTY LEVEL DEFINITIONS

Easy (Shallow or Factual):
- Direct factual lookup or pattern detection.
- Requires no reasoning or inference.
- Example verbs: identify, list, find, describe, summarize.
- Example: "Identify calls where customers asked about refund status."

Medium (Moderate Inference or Single Reasoning Step):
- Requires limited reasoning, correlation, or interpretation.
- May link multiple attributes (e.g., behavior, timing, sentiment).
- Example verbs: analyze, determine, correlate, evaluate.
- Example: "Determine how delay in agent response affects customer satisfaction."

Hard (Complex or Multi-Hop Reasoning):
- Requires deep reasoning, comparison, or hypothetical thinking.
- Often involves counterfactuals, multi-entity or temporal dependencies.
- Example verbs: why, compare, infer, simulate, predict.
- Example: "Why did escalation not occur despite repeated frustration and multiple transfers?"

---

RECLASSIFICATION INSTRUCTIONS

This query received a low calibrated difficulty score ({difficulty_calibrated:.1f}/10) indicating potential misclassification.

Given the query and its originally assigned difficulty:
1. Analyze the reasoning depth and complexity required to answer the query
2. Check for counterfactuals, multi-hop reasoning, or complex comparisons
3. Determine the CORRECT difficulty level according to the strict definitions above
4. Provide clear reasoning for the reclassification

CRITICAL RULES:
- Any counterfactual language ("what if", "would have", "if...had") automatically makes a query HARD
- "Why" questions are typically HARD unless very simple
- Multi-entity comparisons with conditions are usually HARD
- Simple retrievals with no reasoning should be EASY

---

OUTPUT FORMAT

Return your output strictly as JSON in the structure below:

{{
  "query": "<the query>",
  "original_difficulty": "<easy | medium | hard>",
  "reclassified_difficulty": "<easy | medium | hard>",
  "reclassification_reason": "<1–3 sentences explaining why the new label better fits the reasoning complexity>",
  "confidence": "<high|medium|low>",
  "key_complexity_factors": ["<list of complex elements found>"]
}}

---

EXAMPLES:

Example 1:
Query: "What would be the churn intent signal frequency if customers who were offered a fee waiver had not received it..."
Original Difficulty: medium
Difficulty Calibrated: 0.0
JSON Output:
{{
  "query": "What would be the churn intent signal frequency if customers who were offered a fee waiver had not received it...",
  "original_difficulty": "medium",
  "reclassified_difficulty": "hard",
  "reclassification_reason": "Explicit counterfactual reasoning requiring hypothetical scenario analysis across treatment groups - automatic hard difficulty.",
  "confidence": "high",
  "key_complexity_factors": ["counterfactual scenario", "what-if reasoning", "multi-group comparison"]
}}

Example 2:
Query: "What would the first-call resolution rate be if every call that was transferred had been resolved on the first call?"
Original Difficulty: easy
Difficulty Calibrated: 0.0
JSON Output:
{{
  "query": "What would the first-call resolution rate be if every call that was transferred had been resolved on the first call?",
  "original_difficulty": "easy",
  "reclassified_difficulty": "hard",
  "reclassification_reason": "Pure counterfactual simulation of scenarios that never occurred, requiring complex hypothetical reasoning.",
  "confidence": "high",
  "key_complexity_factors": ["counterfactual simulation", "hypothetical scenario"]
}}

Example 3:
Query: "What customer and agent behavioral factors contribute most to refund requests for overcharges..."
Original Difficulty: medium
Difficulty Calibrated: 3.6
JSON Output:
{{
  "query": "What customer and agent behavioral factors contribute most to refund requests for overcharges...",
  "original_difficulty": "medium",
  "reclassified_difficulty": "hard",
  "reclassification_reason": "Multi-factor analysis across customer and agent dimensions with correlation modeling exceeds single-step reasoning.",
  "confidence": "medium",
  "key_complexity_factors": ["multi-entity analysis", "correlation modeling", "behavioral factor ranking"]
}}

---

Now reclassify this low-scoring query:

QUERY: {query}
ORIGINAL DIFFICULTY: {assigned_difficulty}
DIFFICULTY CALIBRATED: {difficulty_calibrated}

Return JSON only.
"""

def call_groq_for_reclassification(prompt):
    """Call Groq API for reclassification"""
    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=1024
        )
        result = response.choices[0].message.content.strip()

        # Parse JSON response
        return json.loads(result)
    except json.JSONDecodeError as e:
        print(f"JSON parsing error: {e}")
        print(f"Raw response: {result}")
        return None
    except Exception as e:
        print(f"Error calling Groq API: {e}")
        return None

def reclassify_low_scoring_queries(input_file="calibrated.json", output_file="reclassified.json", threshold=5.0):
    """Main reclassification function"""

    # Load calibrated results
    with open(input_file, 'r') as f:
        data = json.load(f)

    reclassified_results = []
    reclassified_count = 0

    print(f"Starting reclassification for queries with score < {threshold}")
    print(f"Total queries: {len(data['results'])}")

    for i, query_result in enumerate(data['results']):
        print(f"Processing query {i+1}/{len(data['results'])}: {query_result['difficulty_calibrated']:.1f}")

        if query_result['difficulty_calibrated'] < threshold:
            print(f"  → Low score detected, reclassifying...")

            # Prepare reclassification prompt
            reclass_prompt = RECLASSIFICATION_PROMPT.format(
                query=query_result['query'],
                assigned_difficulty=query_result['difficulty'],
                difficulty_calibrated=query_result['difficulty_calibrated']
            )

            # Call Groq for reclassification
            reclassification = call_groq_for_reclassification(reclass_prompt)

            if reclassification:
                # Update the query result with reclassification info
                query_result.update({
                    'original_difficulty': query_result['difficulty'],
                    'difficulty': reclassification['reclassified_difficulty'],
                    'reclassification_reason': reclassification['reclassification_reason'],
                    'reclassification_confidence': reclassification['confidence'],
                    'key_complexity_factors': reclassification['key_complexity_factors'],
                    'was_reclassified': True
                })
                reclassified_count += 1
                print(f"  → Reclassified: {query_result['original_difficulty']} → {query_result['difficulty']}")
            else:
                query_result['was_reclassified'] = False
                print(f"  → Reclassification failed, keeping original")
        else:
            query_result['was_reclassified'] = False

        reclassified_results.append(query_result)

    # Update metadata
    data['results'] = reclassified_results
    data['metadata']['reclassification_applied'] = True
    data['metadata']['reclassification_threshold'] = threshold
    data['metadata']['reclassified_count'] = reclassified_count
    data['metadata']['total_queries_after_reclassification'] = len(reclassified_results)

    # Save reclassified results
    with open(output_file, 'w') as f:
        json.dump(data, f, indent=2)

    print(f"\nReclassification completed!")
    print(f"Total reclassified: {reclassified_count}")
    print(f"Results saved to: {output_file}")

    return data

def analyze_reclassification_results(output_file="reclassified.json"):
    """Analyze the reclassification results"""
    with open(output_file, 'r') as f:
        data = json.load(f)

    reclassified = [q for q in data['results'] if q.get('was_reclassified', False)]

    print(f"\n--- Reclassification Analysis ---")
    print(f"Total queries: {len(data['results'])}")
    print(f"Reclassified queries: {len(reclassified)}")

    # Count changes by type
    changes = {}
    for q in reclassified:
        change_type = f"{q['original_difficulty']}→{q['difficulty']}"
        changes[change_type] = changes.get(change_type, 0) + 1

    print(f"\nDifficulty changes:")
    for change, count in changes.items():
        print(f"  {change}: {count} queries")

    # Show confidence distribution
    confidences = {}
    for q in reclassified:
        conf = q.get('reclassification_confidence', 'unknown')
        confidences[conf] = confidences.get(conf, 0) + 1

    print(f"\nConfidence levels:")
    for conf, count in confidences.items():
        print(f"  {conf}: {count} queries")

# Run the reclassification
if __name__ == "__main__":
    # Reclassify queries with scores below 5.0
    reclassified_data = reclassify_low_scoring_queries(
        input_file="input.json", #File got after evaluation or own
        output_file="reclassified.json",
        threshold=5.0
    )

    # Analyze results
    analyze_reclassification_results("reclassified.json")